# Rotax 914 ULF - Step 2: Which part is failing?

**In one sentence:** When something is wrong, names the failing component, out of the faults this engine can physically have. Two faults can be present at once.

| Goes in | Comes out |
|---|---|
| The same 128-second window | One probability per fault type |

Starts from what step 1 (*Is anything wrong?*) already learned.

Run the cells top to bottom with **Shift+Enter**. If this step was already trained with `run.py`, training just loads the saved model in a second; otherwise it trains here.

### 0. Setup

In [ ]:
# Finds the project folder wherever Jupyter was started, then loads everything.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers.py')): break
    if os.path.exists(os.path.join(d, 'validation_v4', 'nb_helpers.py')):
        d = os.path.join(d, 'validation_v4'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers import *

ENGINE, PHASE = '914', 'diagnosis'
setup(ENGINE, PHASE)

### 1. Which faults this engine can have
Each fault type is named after the part it damages. Faults for parts the engine
does not have (a 912 has no turbo) never appear, so the model is never asked to
learn them. Grey bars would mean a fault that cannot occur here.

In [ ]:
df = load_rows(ENGINE, ["scenario_id"] + P.FM_COLS, split="train")
per_flight = df.groupby("scenario_id")[P.FM_COLS].max() >= P.DETECTION_THRESHOLD
counts = per_flight.sum().rename(lambda c: c[3:])
possible = applicable_faults(ENGINE)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([pretty(f) for f in counts.index], counts.values,
        color=[FAULTY if f in possible else "#bbb" for f in counts.index])
ax.invert_yaxis(); ax.set_xlabel("training flights (one data file) carrying this fault")
ax.set_title(f"{len(possible)} fault types are possible on this engine")
plt.tight_layout(); plt.show()

### 2. Train the model (or load it)

In [ ]:
payload = train_or_load(ENGINE, PHASE)
print(f"Done. Trained for {payload.get('epochs_run', '?')} passes over the data.")

### 3. Scorecard
Measured on the test flights. The quality bar catches a model that did not learn - it is a safety check, not a target.

In [ ]:
scorecard(ENGINE, PHASE)

### 4. See it working on flights it never saw
For each fault type: red is the share of real cases the model named, blue is
the share of its calls that were right. Both near 1 is ideal.

In [ ]:
r = predict_test(ENGINE, PHASE)
t, p = r["true"] >= P.DETECTION_THRESHOLD, r["pred"] >= 0.5
rows = []
for i, name in enumerate(P.FAULT_MODES):
    if name in applicable_faults(ENGINE) and t[:, i].any():
        hit = (t[:, i] & p[:, i]).sum()
        rows.append({"fault": pretty(name), "caught": hit / t[:, i].sum(),
                     "calls that were right": hit / max(p[:, i].sum(), 1),
                     "test windows": int(t[:, i].sum())})
tab = pd.DataFrame(rows).sort_values("caught")
fig, ax = plt.subplots(figsize=(8.5, 0.4 * len(tab) + 1.2))
yy = np.arange(len(tab))
ax.barh(yy - 0.2, tab["caught"], 0.4, color=FAULTY, label="share of real cases named")
ax.barh(yy + 0.2, tab["calls that were right"], 0.4, color=HEALTHY, label="share of its calls that were right")
ax.set_yticks(yy, tab["fault"]); ax.set_xlim(0, 1); ax.legend(fontsize=8, loc="lower right")
ax.set_title("How well each fault is named"); plt.tight_layout(); plt.show()
tab.round(2)

### 5. What this means

In [ ]:
best, worst = tab.iloc[-1], tab.iloc[0]
print(f"Easiest to name: {best['fault']} - {best['caught']:.0%} of real cases.")
print(f"Hardest to name: {worst['fault']} - {worst['caught']:.0%}.")
if ENGINE != "912":
    print("Turbo and wastegate faults are hard by nature: below the turbo's critical altitude the "
          "wastegate simply closes further and hides them, exactly as on a real engine.")

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - the unit every model looks at. |
| **Twin** | A perfectly healthy simulated copy of the engine, flown through the exact same flight. |
| **Residual** | Real reading minus the healthy twin's reading. Near zero means the engine behaves as it should. |
| **Train / validation / test** | Flights used to learn / to pick the best version / to score it. A flight is only ever in one of them, so the score is on flights the model never saw. |
| **TBO** | Time between overhauls - the engine's scheduled life: 2000 h, or 1200 h for the 915. |
| **Warm start** | A step begins from what an earlier step already learned instead of from nothing. |